# WS02: Data Modeling Approaches

## Overview
This workshop explores two fundamental approaches to data modeling in FIWARE:

1. **Object-Oriented Approach**: Model real-world objects (rooms, sensors, actuators) as individual entities with their own properties and relationships.

2. **Hierarchical Approach**: Model only upper-level objects (e.g., locations/rooms) as entities; represent sensors and actuators as properties/attributes of those entities.

## Learning Objectives
- Understand the pros and cons of each approach
- Implement both models for the intelligent building use case
- Compare ease of specific queries (e.g., reading CO2 levels)
- Evaluate impact of changes (e.g., replacing sensors)
- Assess adaptability to new requirements (e.g., adding occupancy sensors)

## Use Case Reminder

## Use Case
We are digitizing an intelligent building with:
- 1 office building
- 3 floors
- 10 office rooms per floor
- Each office room contains following devices:
| Sensors | Actuators |
| :--- | :--- |
| Temperature sensor | Air ventilation unit |
| Humidity sensor | Radiator thermostat |
| CO2 sensor | Fan coil unit |

## Setup

In [ ]:
# Import required packages
import logging
from typing import Literal, Optional, List
from pydantic import ConfigDict, BaseModel, Field
from pydantic.fields import FieldInfo
from pprint import pprint

from filip.clients.ngsi_v2 import ContextBrokerClient
from filip.models.base import FiwareHeader, DataType
from filip.models.ngsi_v2.context import (
    ContextEntity,
    ContextAttribute,
    NamedContextAttribute,
)
from filip.config import settings

# Configure logging
logging.basicConfig(
    level="INFO",
    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S",
)
logger = logging.getLogger(__name__)

# Configuration
CB_URL = settings.CB_URL
SERVICE = "workshop"
SERVICE_PATH_OO = "/building/object-oriented"
SERVICE_PATH_HIER = "/building/hierarchical"

print("Setup completed successfully!")

## Part 1: Object-Oriented Approach

In this approach, each physical object (room, sensor, actuator) is modeled as a separate entity. This allows fine-grained control and flexibility but requires more entities.

### 1.1 Define Custom Entity Models for Object-Oriented Approach

In [ ]:
# Define a custom Room entity for the object-oriented approach
class Room_OO(ContextEntity):
    """Object-Oriented Room Entity"""
    
    type: str = FieldInfo.merge_field_infos(
        ContextEntity.model_fields["type"],
        default="Room",
        frozen=True,
    )
    
    # Room attributes
    floor: ContextAttribute = ContextAttribute(type="Integer", description="Floor number")
    roomNumber: ContextAttribute = ContextAttribute(type="Integer", description="Room number on the floor")
    location: ContextAttribute = ContextAttribute(type="Text", description="Building location/section")


# Define a custom Temperature Sensor entity
class TemperatureSensor(ContextEntity):
    """Temperature Sensor Entity"""
    
    type: str = FieldInfo.merge_field_infos(
        ContextEntity.model_fields["type"],
        default="TemperatureSensor",
        frozen=True,
    )
    
    temperature: ContextAttribute = ContextAttribute(type="Number", description="Current temperature in Celsius")
    sensorModel: ContextAttribute = ContextAttribute(type="Text", description="Sensor model")
    location: ContextAttribute = ContextAttribute(type="Text", description="Physical location")
    

# Define a custom CO2 Sensor entity
class CO2Sensor(ContextEntity):
    """CO2 Sensor Entity"""
    
    type: str = FieldInfo.merge_field_infos(
        ContextEntity.model_fields["type"],
        default="CO2Sensor",
        frozen=True,
    )
    
    co2Level: ContextAttribute = ContextAttribute(type="Number", description="CO2 level in ppm")
    sensorModel: ContextAttribute = ContextAttribute(type="Text", description="Sensor model")
    location: ContextAttribute = ContextAttribute(type="Text", description="Physical location")


# Define a custom Humidity Sensor entity
class HumiditySensor(ContextEntity):
    """Humidity Sensor Entity"""
    
    type: str = FieldInfo.merge_field_infos(
        ContextEntity.model_fields["type"],
        default="HumiditySensor",
        frozen=True,
    )
    
    humidity: ContextAttribute = ContextAttribute(type="Number", description="Humidity level in percentage")
    sensorModel: ContextAttribute = ContextAttribute(type="Text", description="Sensor model")
    location: ContextAttribute = ContextAttribute(type="Text", description="Physical location")


# Define a custom Actuator entity (e.g., Radiator Thermostat)
class RadiatorThermostat(ContextEntity):
    """Radiator Thermostat Entity"""
    
    type: str = FieldInfo.merge_field_infos(
        ContextEntity.model_fields["type"],
        default="RadiatorThermostat",
        frozen=True,
    )
    
    setpoint: ContextAttribute = ContextAttribute(type="Number", description="Target temperature setpoint")
    power: ContextAttribute = ContextAttribute(type="Boolean", description="Power state")
    location: ContextAttribute = ContextAttribute(type="Text", description="Physical location")

print("Custom entity models defined successfully!")

### 1.2 Create and Populate Object-Oriented Entities

In [ ]:
# Initialize client for object-oriented approach
fiware_header_oo = FiwareHeader(service=SERVICE, service_path=SERVICE_PATH_OO)
cb_client_oo = ContextBrokerClient(url=CB_URL, fiware_header=fiware_header_oo)

# Create room entities
print("Creating rooms (Object-Oriented Approach)...")
rooms_oo = []

for floor in range(1, 4):
    for room_num in range(1, 4):
        room_id = f"urn:ngsi-ld:Room:OO:{floor:02d}{room_num:02d}"
        room = Room_OO(
            id=room_id,
            floor={"value": floor, "type": "Integer"},
            roomNumber={"value": room_num, "type": "Integer"},
            location={"value": f"Section-{floor}", "type": "Text"}
        )
        rooms_oo.append(room)
        try:
            cb_client_oo.post_entity(entity=room)
            print(f"  ✓ Created room: {room_id}")
        except Exception as e:
            print(f"  ✗ Failed to create room: {e}")

print(f"\nTotal rooms created: {len(rooms_oo)}")

### 1.3 Create Sensor Entities (Object-Oriented)

In [ ]:
# Create temperature sensors for each room
print("Creating temperature sensors (Object-Oriented Approach)...")
temp_sensors_oo = []

for floor in range(1, 4):
    for room_num in range(1, 4):
        sensor_id = f"urn:ngsi-ld:TemperatureSensor:OO:{floor:02d}{room_num:02d}"
        sensor = TemperatureSensor(
            id=sensor_id,
            temperature={"value": 20 + floor, "type": "Number"},
            sensorModel={"value": "SHT31-D", "type": "Text"},
            location={"value": f"Room {floor}-{room_num}", "type": "Text"}
        )
        temp_sensors_oo.append(sensor)
        try:
            cb_client_oo.post_entity(entity=sensor)
        except Exception as e:
            print(f"  ✗ Failed to create sensor {sensor_id}: {e}")

print(f"✓ Created {len(temp_sensors_oo)} temperature sensors")

# Create CO2 sensors for each room
print("\nCreating CO2 sensors (Object-Oriented Approach)...")
co2_sensors_oo = []

for floor in range(1, 4):
    for room_num in range(1, 4):
        sensor_id = f"urn:ngsi-ld:CO2Sensor:OO:{floor:02d}{room_num:02d}"
        sensor = CO2Sensor(
            id=sensor_id,
            co2Level={"value": 400 + floor * 100, "type": "Number"},
            sensorModel={"value": "MH-Z19C", "type": "Text"},
            location={"value": f"Room {floor}-{room_num}", "type": "Text"}
        )
        co2_sensors_oo.append(sensor)
        try:
            cb_client_oo.post_entity(entity=sensor)
        except Exception as e:
            print(f"  ✗ Failed to create sensor {sensor_id}: {e}")

print(f"✓ Created {len(co2_sensors_oo)} CO2 sensors")

### 1.4 Query Examples - Object-Oriented Approach

#### Task A: Read CO2 level of one room

In [ ]:
# OBJECT-ORIENTED: Read CO2 level of one room
print("Object-Oriented Approach: Reading CO2 level of Room 1-1")
print("-" * 60)

# Find the CO2 sensor for room 1-1
co2_sensor_id = "urn:ngsi-ld:CO2Sensor:OO:0101"
co2_sensors = cb_client_oo.get_entity_list(entity_ids=[co2_sensor_id])

if co2_sensors:
    sensor = co2_sensors[0]
    co2_level = sensor.get_attribute("co2Level")
    print(f"CO2 Level in Room 1-1: {co2_level.value} ppm")
else:
    print("Sensor not found")

print("\nObservations:")
print("✓ Easy query - directly find the CO2 sensor entity")
print("✓ Sensor is a separate entity with its own properties")
print("✓ Can query sensor metadata (model, location) independently")

#### Task A: Read CO2 levels of ALL rooms

In [ ]:
# OBJECT-ORIENTED: Read CO2 levels of all rooms
print("Object-Oriented Approach: Reading CO2 levels of ALL rooms")
print("-" * 60)

# Query all CO2 sensors
all_co2_sensors = cb_client_oo.get_entity_list(entity_types=["CO2Sensor"])

print(f"Total CO2 sensors found: {len(all_co2_sensors)}")
print("\nCO2 Levels by Room:")
for sensor in sorted(all_co2_sensors, key=lambda s: s.id):
    location = sensor.get_attribute("location")
    co2_level = sensor.get_attribute("co2Level")
    print(f"  {location.value}: {co2_level.value} ppm")

print("\nObservations:")
print("✓ Requires one query to get all CO2 sensor entities")
print("✓ Direct access to CO2 data without nested structures")
print("✓ Easy to iterate and aggregate data")

#### Task B: Replace a defective temperature sensor

In [ ]:
# OBJECT-ORIENTED: Replace a defective temperature sensor
print("Object-Oriented Approach: Replacing defective temperature sensor in Room 1-1")
print("-" * 60)

# Get the current sensor
old_sensor_id = "urn:ngsi-ld:TemperatureSensor:OO:0101"
new_sensor_id = "urn:ngsi-ld:TemperatureSensor:OO:0101-NEW"

print(f"\nStep 1: Remove old sensor {old_sensor_id}")
try:
    cb_client_oo.delete_entity(entity_id=old_sensor_id, entity_type="TemperatureSensor")
    print("  ✓ Old sensor deleted")
except Exception as e:
    print(f"  ✗ Failed to delete: {e}")

print(f"\nStep 2: Create new sensor {new_sensor_id}")
new_sensor = TemperatureSensor(
    id=new_sensor_id,
    temperature={"value": 21.5, "type": "Number"},
    sensorModel={"value": "SHT35-DIS", "type": "Text"},  # Different model
    location={"value": "Room 1-1", "type": "Text"}
)
try:
    cb_client_oo.post_entity(entity=new_sensor)
    print("  ✓ New sensor created")
except Exception as e:
    print(f"  ✗ Failed to create: {e}")

print("\nObservations:")
print("✓ Simple: Delete old entity, create new one")
print("✓ Old sensor metadata is automatically removed")
print("✓ No need to update parent entity")
print("✗ Requires ID change (entity linking must be updated if references exist)")

---

## Part 2: Hierarchical Approach

In this approach, only upper-level objects (rooms) are entities. Sensors and actuators are represented as attributes/properties of the room entity.

### 2.1 Define Custom Entity Model for Hierarchical Approach

In [ ]:
# Define structured models for sensor data
class SensorData(BaseModel):
    """Sensor reading data model"""
    model_config = ConfigDict(populate_by_name=True)
    
    value: float = Field(description="Sensor value")
    unit: str = Field(description="Measurement unit")
    timestamp: str = Field(default="", description="Timestamp of reading")
    sensorModel: str = Field(default="", alias="sensorModel", description="Sensor model")


class TemperatureSensorAttribute(ContextAttribute):
    """Temperature sensor as attribute"""
    type: Literal["StructuredValue"] = FieldInfo.merge_field_infos(
        ContextAttribute.model_fields["type"],
        default="StructuredValue",
        frozen=True,
    )
    value: Optional[dict] = None


class CO2SensorAttribute(ContextAttribute):
    """CO2 sensor as attribute"""
    type: Literal["StructuredValue"] = FieldInfo.merge_field_infos(
        ContextAttribute.model_fields["type"],
        default="StructuredValue",
        frozen=True,
    )
    value: Optional[dict] = None


class HumiditySensorAttribute(ContextAttribute):
    """Humidity sensor as attribute"""
    type: Literal["StructuredValue"] = FieldInfo.merge_field_infos(
        ContextAttribute.model_fields["type"],
        default="StructuredValue",
        frozen=True,
    )
    value: Optional[dict] = None


# Define hierarchical Room entity
class Room_Hierarchical(ContextEntity):
    """Hierarchical Room Entity with sensors as attributes"""
    
    type: str = FieldInfo.merge_field_infos(
        ContextEntity.model_fields["type"],
        default="Room",
        frozen=True,
    )
    
    floor: ContextAttribute = ContextAttribute(type="Integer")
    roomNumber: ContextAttribute = ContextAttribute(type="Integer")
    location: ContextAttribute = ContextAttribute(type="Text")
    
    # Sensors as attributes (hierarchical)
    temperatureSensor: TemperatureSensorAttribute = TemperatureSensorAttribute()
    co2Sensor: CO2SensorAttribute = CO2SensorAttribute()
    humiditySensor: HumiditySensorAttribute = HumiditySensorAttribute()


print("Custom hierarchical entity model defined successfully!")

### 2.2 Create Hierarchical Room Entities

In [ ]:
# Initialize client for hierarchical approach
fiware_header_hier = FiwareHeader(service=SERVICE, service_path=SERVICE_PATH_HIER)
cb_client_hier = ContextBrokerClient(url=CB_URL, fiware_header=fiware_header_hier)

# Create room entities with embedded sensor data
print("Creating rooms with sensors (Hierarchical Approach)...")
rooms_hier = []

for floor in range(1, 4):
    for room_num in range(1, 4):
        room_id = f"urn:ngsi-ld:Room:HIER:{floor:02d}{room_num:02d}"
        
        # Prepare sensor data as structured values
        room = Room_Hierarchical(
            id=room_id,
            floor={"value": floor, "type": "Integer"},
            roomNumber={"value": room_num, "type": "Integer"},
            location={"value": f"Section-{floor}", "type": "Text"},
            temperatureSensor={
                "type": "StructuredValue",
                "value": {
                    "value": 20 + floor,
                    "unit": "Celsius",
                    "sensorModel": "SHT31-D"
                }
            },
            co2Sensor={
                "type": "StructuredValue",
                "value": {
                    "value": 400 + floor * 100,
                    "unit": "ppm",
                    "sensorModel": "MH-Z19C"
                }
            },
            humiditySensor={
                "type": "StructuredValue",
                "value": {
                    "value": 45 + room_num * 2,
                    "unit": "percent",
                    "sensorModel": "SHT31-D"
                }
            }
        )
        
        rooms_hier.append(room)
        try:
            cb_client_hier.post_entity(entity=room)
            print(f"  ✓ Created room with sensors: {room_id}")
        except Exception as e:
            print(f"  ✗ Failed to create room: {e}")

print(f"\nTotal rooms created: {len(rooms_hier)}")

### 2.3 Query Examples - Hierarchical Approach

#### Task A: Read CO2 level of one room

In [ ]:
# HIERARCHICAL: Read CO2 level of one room
print("Hierarchical Approach: Reading CO2 level of Room 1-1")
print("-" * 60)

# Find the room and access CO2 sensor attribute
room_id = "urn:ngsi-ld:Room:HIER:0101"
rooms = cb_client_hier.get_entity_list(entity_ids=[room_id])

if rooms:
    room = rooms[0]
    co2_sensor = room.get_attribute("co2Sensor")
    if co2_sensor and co2_sensor.value:
        co2_data = co2_sensor.value
        print(f"CO2 Level in Room 1-1: {co2_data['value']} {co2_data['unit']}")
        print(f"Sensor Model: {co2_data['sensorModel']}")
else:
    print("Room not found")

print("\nObservations:")
print("✓ One query to get room and all its sensors")
print("✓ All sensor data is in one place (room entity)")
print("✗ Requires nested value access")

#### Task A: Read CO2 levels of ALL rooms

In [ ]:
# HIERARCHICAL: Read CO2 levels of all rooms
print("Hierarchical Approach: Reading CO2 levels of ALL rooms")
print("-" * 60)

# Query all rooms
all_rooms = cb_client_hier.get_entity_list(entity_types=["Room"])

print(f"Total rooms found: {len(all_rooms)}")
print("\nCO2 Levels by Room:")
for room in sorted(all_rooms, key=lambda r: r.id):
    room_info = f"{room.get_attribute('location').value}"
    co2_sensor = room.get_attribute("co2Sensor")
    if co2_sensor and co2_sensor.value:
        co2_value = co2_sensor.value['value']
        print(f"  {room_info}: {co2_value} ppm")

print("\nObservations:")
print("✓ One query returns all rooms and all their sensor data")
print("✓ Natural grouping - sensors are with their room")
print("✓ Reduces number of requests")

#### Task B: Replace a defective temperature sensor

In [ ]:
# HIERARCHICAL: Replace a defective temperature sensor
print("Hierarchical Approach: Replacing defective temperature sensor in Room 1-1")
print("-" * 60)

room_id = "urn:ngsi-ld:Room:HIER:0101"

print(f"\nStep 1: Fetch the room entity")
rooms = cb_client_hier.get_entity_list(entity_ids=[room_id])
if rooms:
    room = rooms[0]
    print(f"  ✓ Room fetched")
    
    print(f"\nStep 2: Update temperature sensor attribute")
    # Get current sensor attribute
    temp_attr = room.get_attribute("temperatureSensor")
    # Update with new sensor data
    temp_attr.value = {
        "value": 21.5,
        "unit": "Celsius",
        "sensorModel": "SHT35-DIS"  # New model
    }
    room.update_attribute([temp_attr])
    
    print(f"\nStep 3: Update room in Context Broker")
    try:
        cb_client_hier.patch_entity(room)
        print(f"  ✓ Temperature sensor updated")
    except Exception as e:
        print(f"  ✗ Failed to update: {e}")
else:
    print("Room not found")

print("\nObservations:")
print("✓ Only one entity update needed")
print("✓ Metadata (sensor model) can be easily versioned")
print("✓ Room context is preserved")
print("✗ Must fetch entire room entity even to change one sensor")

---

## Part 3: Task C - Adding Occupancy Sensors

Now we'll add a new requirement: occupancy sensors in all rooms. Compare how each approach handles this change.

### 3.1 Object-Oriented: Adding Occupancy Sensors

In [ ]:
# Define Occupancy Sensor entity for OO approach
class OccupancySensor(ContextEntity):
    """Occupancy Sensor Entity"""
    
    type: str = FieldInfo.merge_field_infos(
        ContextEntity.model_fields["type"],
        default="OccupancySensor",
        frozen=True,
    )
    
    occupied: ContextAttribute = ContextAttribute(type="Boolean", description="Occupancy status")
    personCount: ContextAttribute = ContextAttribute(type="Integer", description="Number of people detected")
    sensorModel: ContextAttribute = ContextAttribute(type="Text", description="Sensor model")
    location: ContextAttribute = ContextAttribute(type="Text", description="Physical location")

print("Object-Oriented: Adding occupancy sensors...")
occupancy_sensors_oo = []

for floor in range(1, 4):
    for room_num in range(1, 4):
        sensor_id = f"urn:ngsi-ld:OccupancySensor:OO:{floor:02d}{room_num:02d}"
        sensor = OccupancySensor(
            id=sensor_id,
            occupied={"value": room_num % 2 == 0, "type": "Boolean"},
            personCount={"value": room_num if room_num % 2 == 0 else 0, "type": "Integer"},
            sensorModel={"value": "PIR-HC-SR501", "type": "Text"},
            location={"value": f"Room {floor}-{room_num}", "type": "Text"}
        )
        occupancy_sensors_oo.append(sensor)
        try:
            cb_client_oo.post_entity(entity=sensor)
        except Exception as e:
            print(f"  ✗ Failed to create sensor {sensor_id}: {e}")

print(f"✓ Created {len(occupancy_sensors_oo)} occupancy sensors")
print("\nObject-Oriented Observations:")
print("✓ Simple: Just create new sensor entities")
print("✓ No changes needed to room entities")
print("✓ Can query occupancy independently")
print("✗ Existing room entities don't know about occupancy until explicit query")

### 3.2 Hierarchical: Adding Occupancy Sensors

In [ ]:
# Define occupancy sensor attribute for hierarchical approach
class OccupancySensorAttribute(ContextAttribute):
    """Occupancy sensor as attribute"""
    type: Literal["StructuredValue"] = FieldInfo.merge_field_infos(
        ContextAttribute.model_fields["type"],
        default="StructuredValue",
        frozen=True,
    )
    value: Optional[dict] = None

print("Hierarchical: Adding occupancy sensors...")

print("\nStep 1: Fetch all rooms and add occupancy sensor attribute")
rooms_hier_all = cb_client_hier.get_entity_list(entity_types=["Room"])
print(f"Fetched {len(rooms_hier_all)} rooms")

for room in rooms_hier_all:
    # Create occupancy sensor attribute
    occupancy_attr = OccupancySensorAttribute(
        value={
            "occupied": room.id.endswith("02"),  # Example: room 2, 5, 8... are occupied
            "personCount": 1 if room.id.endswith("02") else 0,
            "sensorModel": "PIR-HC-SR501"
        }
    )
    
    # Add attribute to room
    room.add_attributes([occupancy_attr])
    
    try:
        cb_client_hier.patch_entity(room)
    except Exception as e:
        print(f"  ✗ Failed to update room: {e}")

print(f"✓ Updated {len(rooms_hier_all)} rooms with occupancy sensor")

print("\nHierarchical Observations:")
print("✓ All sensor data stays together with the room")
print("✓ One query returns complete room information")
print("✗ Must fetch and update all room entities")
print("✗ Larger entities with more nested data")

---

## Part 4: Comparison Summary

Let's create a comprehensive comparison.

In [ ]:
import pandas as pd

# Create comparison table
comparison_data = {
    'Criterion': [
        'Number of Entities',
        'Entity Size',
        'Data Redundancy',
        'Query Simplicity (single room)',
        'Query Simplicity (all rooms)',
        'Sensor Replacement',
        'Adding New Sensor Type',
        'Update Scope',
        'Semantic Clarity',
        'Performance (number of requests)',
    ],
    'Object-Oriented': [
        'Many (room + all sensors)',
        'Small (focused)',
        'Location info in each sensor',
        'Multiple queries needed',
        'Easy - query by sensor type',
        'Delete old, create new entity',
        'Just create new sensor entities',
        'Individual sensor',
        'Very clear - each entity is one thing',
        'Higher (1 query per entity type)',
    ],
    'Hierarchical': [
        'Few (room only)',
        'Large (room + all sensors)',
        'Location info only in room',
        'One query to get room + all sensors',
        'One query returns all data',
        'Update room entity attribute',
        'Update all room entities',
        'Entire room entity',
        'Natural grouping, but sensors are properties',
        'Lower (fewer entities to query)',
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "="*100)
print("COMPARISON: Object-Oriented vs Hierarchical Data Modeling")
print("="*100)
print(df_comparison.to_string(index=False))
print("="*100)

## Part 5: Recommendations

### When to Use Object-Oriented Approach:

1. **Sensor-centric applications**: Where sensors are managed independently
2. **High-frequency sensor updates**: Individual sensors can be updated without touching room data
3. **Complex sensor ecosystems**: Many different sensor types with different update frequencies
4. **Reusable sensors**: Sensors can be moved between rooms
5. **Fine-grained permissions**: Can assign different permissions to room vs. sensor access

**Example Use Cases**: IoT platforms, sensor networks, industrial monitoring

### When to Use Hierarchical Approach:

1. **Building management**: Rooms as primary entities with integrated sensors
2. **Real-time dashboards**: Need all room data in one query
3. **Energy management**: Focus on room-level consumption
4. **Limited sensor changes**: Sensors are mostly stable within a room
5. **Reduced query complexity**: Minimize number of API requests

**Example Use Cases**: Smart buildings, facility management, comfort control systems

### Hybrid Approach:

For complex systems, consider a **hybrid approach**:
- Rooms as hierarchical entities (with commonly accessed sensors as attributes)
- Specialized or frequently-updated sensors as separate entities
- Use relationships/links between room and sensor entities

## Exercises

1. **Extend the Hierarchical Model**: Add actuators (radiator thermostat, fan coil unit) to the hierarchical room entity.

2. **Query Performance**: Compare the number of API calls needed to:
   - Get all sensor readings for all rooms
   - Update all CO2 sensors
   - Find rooms with occupancy and high CO2

3. **Schema Validation**: Implement JSON Schema validation for both approaches using pydantic models.

4. **Data Aggregation**: Write functions to:
   - Calculate average temperature per floor
   - Find rooms with poor air quality (high CO2 + high occupancy)
   - Generate occupancy reports

5. **Migration**: Design a process to migrate from hierarchical to object-oriented model (or vice versa) without data loss.